# Phase A3 — Eval (post-training, full Fold 3)

**Goal:** Eval A3 TypeHead trên **full Fold 3** (2722 patches) —
đồng bộ scope với A2 eval, paper-grade evidence cho Section 4.3.

**Khác A3 training notebook (eval block 500 Fold 1):**
- Full Fold 3 (2722 patches) thay vì 500 Fold 1
- Test fold (Fold 3) — không leak với train fold (Fold 1+2)
- Dead class có thật instances thay vì 0

**Input:**
- LoRA checkpoint `sam3_lora_rank16_final.pt` (Phase A2)
- TypeHead checkpoint `type_head_final.pt` (Phase A3)
- PanNuke Fold 3

**Prerequisites Kaggle:**
1. GPU: T4 (12h session, sẽ dùng ~2h)
2. Datasets:
   - `hipinhththu/pannuke`
   - `hipinhththu/sam3-native-pt`
   - `hipinhththu/phase-a2-lora-weights`
   - `hipinhththu/phase-a3-typehead-weights`

**Compute budget:** ~2h trên T4 (1 backbone + 1 prompt + N type forwards / image).

## 00 — Setup

In [ ]:
import subprocess, sys, os, platform, time, json
print("Python  :", sys.version.split()[0])
print("Platform:", platform.platform())
import torch
print("Torch   :", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU     :", torch.cuda.get_device_name(0))
    print("VRAM    :", torch.cuda.get_device_properties(0).total_memory / 1e9, "GB")

In [ ]:
WORK = "/kaggle/working"
REPO_DIR = f"{WORK}/sam3_research"
SAM3_DIR = f"{REPO_DIR}/sam3"
CHECKPOINT_PATH = "/kaggle/input/datasets/hipinhththu/sam3-native-pt/sam3.pt"
DATA_ROOT = "/kaggle/input/datasets/hipinhththu/pannuke"

LORA_CKPT_CANDIDATES = [
    "/kaggle/input/datasets/hipinhththu/phase-a2-lora-weights/sam3_lora_rank16_final.pt",
    "/kaggle/input/phase-a2-lora-weights/sam3_lora_rank16_final.pt",
    f"{WORK}/sam3_lora_rank16_final.pt",
]
LORA_CKPT_PATH = next((p for p in LORA_CKPT_CANDIDATES if os.path.exists(p)), None)
assert LORA_CKPT_PATH, f"Khong tim thay LoRA. Da check: {LORA_CKPT_CANDIDATES}"
print(f"LoRA   : {LORA_CKPT_PATH}")

TYPEHEAD_CKPT_CANDIDATES = [
    "/kaggle/input/datasets/hipinhththu/phase-a3-typehead-weights/type_head_final.pt",
    "/kaggle/input/datasets/hipinhththu/phase-a3-typehead-weights/type_head_epoch3.pt",
    "/kaggle/input/datasets/hipinhththu/phase-a3-typehead-weights/type_head_epoch2.pt",
    "/kaggle/input/phase-a3-typehead-weights/type_head_final.pt",
    f"{WORK}/type_head_final.pt",
]
TYPEHEAD_CKPT_PATH = next((p for p in TYPEHEAD_CKPT_CANDIDATES if os.path.exists(p)), None)
assert TYPEHEAD_CKPT_PATH, f"Khong tim thay TypeHead. Da check: {TYPEHEAD_CKPT_CANDIDATES}"
print(f"TypeHead: {TYPEHEAD_CKPT_PATH}")

if not os.path.exists(REPO_DIR):
    subprocess.run(["git", "clone",
                    "https://github.com/duonguwu/sam3_research.git", REPO_DIR],
                   check=True)
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=False)

assert os.path.exists(CHECKPOINT_PATH), "Attach hipinhththu/sam3-native-pt"
assert os.path.exists(DATA_ROOT), "Attach hipinhththu/pannuke"

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", SAM3_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "scikit-image", "scikit-learn", "matplotlib", "opencv-python",
                "pycocotools", "einops", "tqdm"], check=True)
print("OK setup")

## Helper modules (writefile)

In [ ]:
%%writefile pannuke_loader.py
from __future__ import annotations
from pathlib import Path
from typing import Iterator, List, Optional, Tuple
import numpy as np

CELL_TYPES: List[str] = [
    "Neoplastic", "Inflammatory", "Connective", "Dead", "Epithelial",
]

DEFAULT_ROOT = Path("/kaggle/input/datasets/hipinhththu/pannuke")

def _to_uint8(arr: np.ndarray) -> np.ndarray:
    if arr.dtype == np.uint8:
        return arr
    if arr.max() <= 1.0:
        return (arr * 255).round().clip(0, 255).astype(np.uint8)
    return arr.clip(0, 255).astype(np.uint8)

class PanNukeFold:

    def __init__(self, root, fold: int):
        self.root = Path(root)
        self.fold = fold
        sub = f"Fold {fold}"
        f = f"fold{fold}"
        candidates = [
            self.root / f / sub,   
            self.root / sub,        
        ]
        base = next((c for c in candidates if (c / "images" / f / "images.npy").exists()),
                    None)
        if base is None:
            raise FileNotFoundError(
                f"Không tìm thấy Fold {fold} ở: " + " hoặc ".join(str(c) for c in candidates)
            )
        self.images_path = base / "images" / f / "images.npy"
        self.masks_path  = base / "masks"  / f / "masks.npy"

        
        type_candidates = [
            base / "images" / f / "types.npy",   
            base / "types" / f / "types.npy",     
            base / "types.npy",
            self.root / f / "types.npy",
            self.root / "types" / f / "types.npy",
        ]
        self.types_path = next((p for p in type_candidates if p.exists()), None)

        for p in (self.images_path, self.masks_path):
            if not p.exists():
                raise FileNotFoundError(f"Missing: {p}")

        self.images = np.load(self.images_path, mmap_mode="r")
        self.masks  = np.load(self.masks_path,  mmap_mode="r")
        if self.types_path is not None:
            self.tissue_types = np.load(self.types_path, allow_pickle=True)
        else:
            n = self.images.shape[0]
            self.tissue_types = np.array(["unknown"] * n, dtype=object)
            print(f"[Fold {fold}] WARN: types.npy không tìm thấy ở:")
            for c in type_candidates:
                print(f"  - {c}")
            print(f"[Fold {fold}] Dùng placeholder 'unknown' cho {n} ảnh.")

        assert self.images.shape[0] == self.masks.shape[0] == self.tissue_types.shape[0]
        assert self.images.shape[1:] == (256, 256, 3), f"unexpected: {self.images.shape}"
        assert self.masks.shape[1:]  == (256, 256, 6), f"unexpected: {self.masks.shape}"

    def __len__(self) -> int:
        return self.images.shape[0]

    def __getitem__(self, idx: int) -> dict:
        img = _to_uint8(np.array(self.images[idx]))
        m_all = np.array(self.masks[idx], dtype=np.int32)
        masks_per_type = m_all[..., :5].transpose(2, 0, 1)
        counts = np.array(
            [int(np.unique(masks_per_type[k]).size - 1) for k in range(5)],
            dtype=np.int32,
        )
        return {
            "image": img,
            "masks": masks_per_type,
            "counts": counts,
            "tissue": str(self.tissue_types[idx]),
            "fold": self.fold,
            "idx": idx,
        }

    def iter_samples(self, start: int = 0, end: Optional[int] = None) -> Iterator[dict]:
        end = end or len(self)
        for i in range(start, end):
            yield self[i]

def load_all_folds(root=DEFAULT_ROOT) -> Tuple[PanNukeFold, PanNukeFold, PanNukeFold]:
    root = Path(root)
    return PanNukeFold(root, 1), PanNukeFold(root, 2), PanNukeFold(root, 3)

In [ ]:
%%writefile metrics.py
from __future__ import annotations
from typing import Sequence
import numpy as np

def binary_iou(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return float(inter) / float(union) if union > 0 else 0.0

def binary_dice(a: np.ndarray, b: np.ndarray) -> float:
    a = a.astype(bool); b = b.astype(bool)
    inter = np.logical_and(a, b).sum()
    sa, sb = a.sum(), b.sum()
    return float(2 * inter) / float(sa + sb) if (sa + sb) > 0 else 0.0

def match_pred_to_gt(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray],
                     iou_thresh: float = 0.5) -> dict:
    if not pred_masks and not gt_masks:
        return {"tp": 0, "fp": 0, "fn": 0, "mean_iou": 0.0}
    if not pred_masks:
        return {"tp": 0, "fp": 0, "fn": len(gt_masks), "mean_iou": 0.0}
    if not gt_masks:
        return {"tp": 0, "fp": len(pred_masks), "fn": 0, "mean_iou": 0.0}

    iou_matrix = np.zeros((len(pred_masks), len(gt_masks)), dtype=np.float32)
    for i, pm in enumerate(pred_masks):
        for j, gm in enumerate(gt_masks):
            iou_matrix[i, j] = binary_iou(pm, gm)

    matched_pred, matched_gt = set(), set()
    ious = []
    pairs = np.dstack(np.unravel_index(np.argsort(-iou_matrix.ravel()), iou_matrix.shape))[0]
    for i, j in pairs:
        if iou_matrix[i, j] < iou_thresh:
            break
        if i in matched_pred or j in matched_gt:
            continue
        matched_pred.add(int(i)); matched_gt.add(int(j))
        ious.append(float(iou_matrix[i, j]))

    tp = len(matched_pred)
    fp = len(pred_masks) - tp
    fn = len(gt_masks)  - len(matched_gt)
    return {"tp": tp, "fp": fp, "fn": fn,
            "mean_iou": float(np.mean(ious)) if ious else 0.0}

def panoptic_quality(stats: dict) -> dict:
    tp, fp, fn = stats["tp"], stats["fp"], stats["fn"]
    sq = stats["mean_iou"]
    denom = tp + 0.5 * fp + 0.5 * fn
    rq = tp / denom if denom > 0 else 0.0
    pq = sq * rq
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    return {"PQ": pq, "SQ": sq, "RQ": rq, "F1": f1, "P": precision, "R": recall}

def aggregate_iou_image(pred_masks: Sequence[np.ndarray], gt_masks: Sequence[np.ndarray]) -> float:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu)

def aggregate_iou_dice_image(pred_masks: Sequence[np.ndarray],
                              gt_masks: Sequence[np.ndarray]) -> tuple:
    H, W = (gt_masks[0].shape if gt_masks else
            (pred_masks[0].shape if pred_masks else (256, 256)))
    pu = np.zeros((H, W), dtype=bool)
    for m in pred_masks: pu |= m.astype(bool)
    gu = np.zeros((H, W), dtype=bool)
    for m in gt_masks:   gu |= m.astype(bool)
    return binary_iou(pu, gu), binary_dice(pu, gu)

def union_masks(masks: Sequence[np.ndarray], shape=(256, 256)) -> np.ndarray:
    u = np.zeros(shape, dtype=bool)
    for m in masks:
        u |= m.astype(bool)
    return u.astype(np.uint8)

class ClassWiseAccumulator:

    def __init__(self, class_names):
        self.class_names = list(class_names)
        self.tp = {c: 0 for c in self.class_names}
        self.fp = {c: 0 for c in self.class_names}
        self.fn = {c: 0 for c in self.class_names}

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray, class_name: str):
        p = pred_mask.astype(bool)
        g = gt_mask.astype(bool)
        self.tp[class_name] += int(np.logical_and(p, g).sum())
        self.fp[class_name] += int(np.logical_and(p, np.logical_not(g)).sum())
        self.fn[class_name] += int(np.logical_and(np.logical_not(p), g).sum())

    def class_iou(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = tp + fp + fn
        return float(tp) / float(denom) if denom > 0 else 0.0

    def class_dice(self, class_name: str) -> float:
        tp, fp, fn = self.tp[class_name], self.fp[class_name], self.fn[class_name]
        denom = 2 * tp + fp + fn
        return float(2 * tp) / float(denom) if denom > 0 else 0.0

    def mIoU(self) -> float:
        return float(np.mean([self.class_iou(c) for c in self.class_names]))

    def mDice(self) -> float:
        return float(np.mean([self.class_dice(c) for c in self.class_names]))

    def summary(self) -> dict:
        per_class = {c: {"IoU": self.class_iou(c), "Dice": self.class_dice(c),
                          "TP": self.tp[c], "FP": self.fp[c], "FN": self.fn[c]}
                      for c in self.class_names}
        return {
            "mIoU": self.mIoU(),
            "mDice": self.mDice(),
            "per_class": per_class,
        }

class PerPromptClassAccumulator:

    def __init__(self, class_names, prompts_per_class):
        self.class_names = list(class_names)
        self.prompts_per_class = {c: list(prompts_per_class[c]) for c in self.class_names}
        
        self.accs = {}
        for c, prompts in self.prompts_per_class.items():
            for p in prompts:
                self.accs[(c, p)] = ClassWiseAccumulator([c])

    def update(self, pred_mask: np.ndarray, gt_mask: np.ndarray,
               class_name: str, prompt: str):
        self.accs[(class_name, prompt)].update(pred_mask, gt_mask, class_name)

    def summary(self) -> dict:
        per_class = {}
        for c in self.class_names:
            per_prompt = []
            for p in self.prompts_per_class[c]:
                acc = self.accs[(c, p)]
                per_prompt.append({
                    "prompt": p,
                    "IoU": acc.class_iou(c),
                    "Dice": acc.class_dice(c),
                    "TP": acc.tp[c], "FP": acc.fp[c], "FN": acc.fn[c],
                })
            ious = [pp["IoU"] for pp in per_prompt]
            dices = [pp["Dice"] for pp in per_prompt]
            per_class[c] = {
                "IoU": float(np.mean(ious)),   
                "Dice": float(np.mean(dices)),
                "per_prompt": per_prompt,
            }
        mIoU = float(np.mean([per_class[c]["IoU"] for c in self.class_names]))
        mDice = float(np.mean([per_class[c]["Dice"] for c in self.class_names]))
        return {"mIoU": mIoU, "mDice": mDice, "per_class": per_class}

def bootstrap_ci(values, n_boot: int = 1000, alpha: float = 0.05, seed: int = 0):
    if len(values) == 0:
        return 0.0, 0.0
    rng = np.random.default_rng(seed)
    vals = np.asarray(values, dtype=np.float64)
    boots = [rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)]
    lo = float(np.percentile(boots, 100 * alpha / 2))
    hi = float(np.percentile(boots, 100 * (1 - alpha / 2)))
    return lo, hi

In [ ]:
%%writefile lora_sam3.py
from __future__ import annotations
import math
from typing import Iterable, List, Optional, Set, Tuple
import torch
import torch.nn as nn

class LoRALinear(nn.Module):

    def __init__(self, base: nn.Linear, r: int = 16, alpha: int = 32,
                 dropout: float = 0.05):
        super().__init__()
        self.base = base
        
        for p in self.base.parameters():
            p.requires_grad = False

        self.r = r
        self.alpha = alpha
        self.scaling = alpha / r

        in_f = base.in_features
        out_f = base.out_features
        self.lora_A = nn.Parameter(torch.zeros(r, in_f))
        self.lora_B = nn.Parameter(torch.zeros(out_f, r))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        

        self.dropout = nn.Dropout(dropout) if dropout > 0 else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        out = self.base(x)
        lora_out = self.dropout(x) @ self.lora_A.T @ self.lora_B.T
        return out + lora_out * self.scaling

    @property
    def in_features(self) -> int:
        return self.base.in_features

    @property
    def out_features(self) -> int:
        return self.base.out_features

DEFAULT_LORA_TARGETS: Set[str] = {
    
    
    
    "linear1", "linear2",
}

def inject_lora(
    model: nn.Module,
    target_module_names: Iterable[str] = DEFAULT_LORA_TARGETS,
    r: int = 16,
    alpha: int = 32,
    dropout: float = 0.05,
    path_must_contain: str = "decoder",
    verbose: bool = True,
) -> Tuple[List[str], int]:
    target_set = set(target_module_names)
    replaced: List[str] = []

    for parent_name, parent in list(model.named_modules()):
        
        if path_must_contain and path_must_contain not in parent_name:
            continue
        for attr_name, child in list(parent.named_children()):
            if attr_name in target_set and isinstance(child, nn.Linear):
                lora_layer = LoRALinear(child, r=r, alpha=alpha, dropout=dropout)
                
                base_device = next(child.parameters()).device
                lora_layer.lora_A.data = lora_layer.lora_A.data.to(base_device)
                lora_layer.lora_B.data = lora_layer.lora_B.data.to(base_device)
                setattr(parent, attr_name, lora_layer)
                full_path = f"{parent_name}.{attr_name}" if parent_name else attr_name
                replaced.append(full_path)

    n_lora_params = sum(p.numel() for p in model.parameters()
                        if p.requires_grad)

    if verbose:
        print(f"Injected LoRA into {len(replaced)} modules.")
        print(f"LoRA trainable params: {n_lora_params:,} "
              f"({n_lora_params/1e6:.2f}M)")
        if len(replaced) > 0:
            print("Sample paths:")
            for p in replaced[:5]:
                print(f"  {p}")
            if len(replaced) > 5:
                print(f"  ... ({len(replaced)-5} more)")

    return replaced, n_lora_params

def freeze_non_lora(model: nn.Module) -> Tuple[int, int]:
    n_train = 0
    n_total = 0
    for name, p in model.named_parameters():
        n_total += p.numel()
        if "lora_A" in name or "lora_B" in name:
            p.requires_grad = True
            n_train += p.numel()
        else:
            p.requires_grad = False
    return n_train, n_total

def save_lora_state(model: nn.Module, path: str):
    state = {n: p.detach().cpu()
             for n, p in model.named_parameters()
             if ("lora_A" in n or "lora_B" in n)}
    torch.save(state, path)
    return len(state)

def load_lora_state(model: nn.Module, path: str) -> int:
    state = torch.load(path, map_location="cpu")
    own = dict(model.named_parameters())
    n_loaded = 0
    for k, v in state.items():
        if k in own:
            own[k].data = v.to(own[k].device)
            n_loaded += 1
    return n_loaded

In [ ]:
%%writefile sam3_train.py
from __future__ import annotations
from typing import Dict, List, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.transforms import v2

def make_transform(resolution: int = 1008):
    return v2.Compose([
        v2.ToDtype(torch.uint8, scale=True),
        v2.Resize(size=(resolution, resolution)),
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
    ])

@torch.no_grad()
def encode_image_frozen(model, transform, pil_img, device: str = "cuda"):
    image = v2.functional.to_image(pil_img).to(device)
    image = transform(image).unsqueeze(0)
    with torch.autocast(device_type=device, dtype=torch.bfloat16):
        backbone_out = model.backbone.forward_image(image)

        
        inst_pred = getattr(model, "inst_interactive_predictor", None)
        if inst_pred is not None and "sam2_backbone_out" in backbone_out:
            sam2_bb = backbone_out["sam2_backbone_out"]
            sam2_bb["backbone_fpn"][0] = (
                inst_pred.model.sam_mask_decoder.conv_s0(sam2_bb["backbone_fpn"][0])
            )
            sam2_bb["backbone_fpn"][1] = (
                inst_pred.model.sam_mask_decoder.conv_s1(sam2_bb["backbone_fpn"][1])
            )
    return backbone_out

def encode_text(model, prompt: str, device: str = "cuda"):
    with torch.no_grad():
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            return model.backbone.forward_text([prompt], device=device)

def forward_decoder_with_grad(model, backbone_out: Dict, find_stage,
                              geometric_prompt, device: str = "cuda") -> Dict:
    was_training = model.training
    if was_training:
        model.eval()
    try:
        with torch.autocast(device_type=device, dtype=torch.bfloat16):
            outputs = model.forward_grounding(
                backbone_out=backbone_out,
                find_input=find_stage,
                geometric_prompt=geometric_prompt,
                find_target=None,
            )
    finally:
        if was_training:
            model.train()
    return outputs

def semantic_union_mask(outputs: Dict, target_hw: Tuple[int, int]) -> torch.Tensor:
    
    pred_logits = outputs["pred_logits"].float()         
    pred_masks  = outputs["pred_masks"].float()          
    pres_logit  = outputs["presence_logit_dec"].float()  

    
    cls_prob = pred_logits.sigmoid()                   
    pres     = pres_logit.sigmoid().unsqueeze(1)       
    prob     = (cls_prob * pres).squeeze(-1)           

    
    masks = F.interpolate(
        pred_masks, size=target_hw, mode="bilinear", align_corners=False
    ).sigmoid()  

    
    weighted = prob[:, :, None, None] * masks   

    
    
    
    eps = 1e-4
    log_one_minus = torch.log(torch.clamp(1.0 - weighted, min=eps, max=1.0 - eps))
    log_prod = log_one_minus.sum(dim=1)         
    union = 1.0 - torch.exp(torch.clamp(log_prod, min=-20.0))  
    union = torch.clamp(union, min=eps, max=1.0 - eps)  
    return union.squeeze(0)                     

def dice_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    pred = pred.float()
    target = target.float()
    inter = (pred * target).sum()
    return 1.0 - (2.0 * inter + eps) / (pred.sum() + target.sum() + eps)

def bce_loss(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-4) -> torch.Tensor:
    pred = pred.float()
    target = target.float()
    p = torch.clamp(pred, eps, 1 - eps)
    return -(target * torch.log(p) + (1 - target) * torch.log(1 - p)).mean()

def semantic_seg_loss(pred: torch.Tensor, target: torch.Tensor,
                      bce_weight: float = 0.5,
                      dice_weight: float = 1.0) -> Tuple[torch.Tensor, Dict[str, float]]:
    bce = bce_loss(pred, target)
    dice = dice_loss(pred, target)
    total = bce_weight * bce + dice_weight * dice
    return total, {"bce": float(bce.item()), "dice": float(dice.item()),
                   "loss": float(total.item())}

@torch.no_grad()
def inference_to_binary(outputs: Dict, target_hw: Tuple[int, int],
                        score_threshold: float = 0.3) -> torch.Tensor:
    pred_logits = outputs["pred_logits"]
    pred_masks  = outputs["pred_masks"]
    pres_logit  = outputs["presence_logit_dec"]

    cls_prob = pred_logits.sigmoid()
    pres = pres_logit.sigmoid().unsqueeze(1)
    prob = (cls_prob * pres).squeeze(-1).squeeze(0)  

    masks = F.interpolate(
        pred_masks, size=target_hw, mode="bilinear", align_corners=False
    ).sigmoid().squeeze(0)  

    keep = prob > score_threshold
    if keep.sum() == 0:
        return torch.zeros(target_hw, dtype=torch.bool, device=pred_logits.device)

    selected = (masks[keep] > 0.5)
    union = selected.any(dim=0)
    return union

In [ ]:
%%writefile type_head.py
from __future__ import annotations
from typing import Dict, List, Optional, Tuple
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np

class TypeHead(nn.Module):
    def __init__(self, in_dim: int = 256, hidden_dim: int = 128,
                 num_classes: int = 5, dropout: float = 0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

def roi_pool_feature(backbone_features: torch.Tensor,
                     mask: torch.Tensor) -> torch.Tensor:
    if backbone_features.dim() == 4:
        backbone_features = backbone_features[0]  
    D, Hf, Wf = backbone_features.shape

    
    mask_resized = F.interpolate(
        mask.float()[None, None],
        size=(Hf, Wf),
        mode='bilinear',
        align_corners=False,
    )[0, 0]  

    
    mass = mask_resized.sum()
    if mass < 1e-3:
        return backbone_features.mean(dim=(1, 2))  
    pooled = (backbone_features * mask_resized).sum(dim=(1, 2)) / mass
    return pooled  

def compute_iou_matrix(pred_masks: List[np.ndarray],
                       gt_masks: List[np.ndarray]) -> np.ndarray:
    N_p, N_g = len(pred_masks), len(gt_masks)
    iou_matrix = np.zeros((N_p, N_g), dtype=np.float32)
    for i, p in enumerate(pred_masks):
        for j, g in enumerate(gt_masks):
            inter = np.logical_and(p, g).sum()
            union = np.logical_or(p, g).sum()
            iou_matrix[i, j] = inter / (union + 1e-6) if union > 0 else 0.0
    return iou_matrix

def hungarian_match(iou_matrix: np.ndarray,
                    iou_thresh: float = 0.3) -> List[Tuple[int, int]]:
    try:
        from scipy.optimize import linear_sum_assignment
        
        row_ind, col_ind = linear_sum_assignment(-iou_matrix)
        matches = []
        for r, c in zip(row_ind, col_ind):
            if iou_matrix[r, c] >= iou_thresh:
                matches.append((int(r), int(c)))
        return matches
    except ImportError:
        
        matches = []
        used_g = set()
        
        flat = [(iou_matrix[i, j], i, j)
                for i in range(iou_matrix.shape[0])
                for j in range(iou_matrix.shape[1])]
        flat.sort(reverse=True)
        used_p = set()
        for iou, i, j in flat:
            if iou < iou_thresh:
                break
            if i in used_p or j in used_g:
                continue
            matches.append((i, j))
            used_p.add(i)
            used_g.add(j)
        return matches

def extract_gt_instances(sample: dict, cell_types: List[str]
                         ) -> Tuple[List[np.ndarray], List[int]]:
    masks_per_type = sample["masks"]
    gt_masks = []
    gt_classes = []
    for type_idx in range(5):
        inst_ids = np.unique(masks_per_type[type_idx])
        for inst_id in inst_ids:
            if inst_id == 0:
                continue
            mask = (masks_per_type[type_idx] == inst_id).astype(bool)
            if mask.sum() < 5:  
                continue
            gt_masks.append(mask)
            gt_classes.append(type_idx)
    return gt_masks, gt_classes

def per_class_counts(pred_scores: np.ndarray,
                     pred_probs: np.ndarray) -> np.ndarray:
    counts = (pred_scores[:, None] * pred_probs).sum(axis=0)  
    return counts

def per_class_variance(pred_scores: np.ndarray,
                       pred_probs: np.ndarray) -> np.ndarray:
    weighted = pred_scores[:, None] * pred_probs  
    var = (weighted * (1.0 - weighted)).sum(axis=0)
    return var

In [ ]:
import sys
for p in [".", "/kaggle/working", SAM3_DIR]:
    if p not in sys.path:
        sys.path.insert(0, p)

from pannuke_loader import PanNukeFold, CELL_TYPES, DEFAULT_ROOT
from lora_sam3 import (inject_lora, freeze_non_lora, load_lora_state,
                       DEFAULT_LORA_TARGETS)
from sam3_train import (make_transform, encode_image_frozen, encode_text,
                        forward_decoder_with_grad, inference_to_binary)
from type_head import (TypeHead, roi_pool_feature, compute_iou_matrix,
                       hungarian_match, extract_gt_instances,
                       per_class_counts, per_class_variance)
print("Helpers loaded.")

## 01 — Build SAM3 + Load LoRA + Load TypeHead

In [ ]:
from sam3.model_builder import build_sam3_image_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Build SAM3...")
model = build_sam3_image_model(
    device=device, eval_mode=True,
    checkpoint_path=CHECKPOINT_PATH, load_from_HF=False,
)
model.eval()
print(f"SAM3 params: {sum(p.numel() for p in model.parameters())/1e6:.1f}M")

LORA_R = 16
LORA_ALPHA = 32
replaced, n_lora = inject_lora(
    model, target_module_names=DEFAULT_LORA_TARGETS,
    r=LORA_R, alpha=LORA_ALPHA, dropout=0.0,
)
n_loaded = load_lora_state(model, LORA_CKPT_PATH)
print(f"LoRA inject: {len(replaced)} modules, {n_loaded} tensors loaded")

for p in model.parameters():
    p.requires_grad = False
model.eval()

type_head = TypeHead(in_dim=256, hidden_dim=128, num_classes=5, dropout=0.0).to(device)
state = torch.load(TYPEHEAD_CKPT_PATH, map_location=device)
type_head.load_state_dict(state)
type_head.eval()
n_th = sum(p.numel() for p in type_head.parameters())
print(f"TypeHead loaded: {n_th:,} params ({n_th/1e3:.1f}K)")
print(f"  Source: {TYPEHEAD_CKPT_PATH}")
print("Model fully ready (frozen, eval mode).")

## 02 — Load full Fold 3

In [ ]:
import numpy as np
from PIL import Image
from tqdm import tqdm
import torch.nn.functional as F

np.random.seed(42)
torch.manual_seed(42)

fold3 = PanNukeFold(DEFAULT_ROOT, 3)
print(f"Fold 3: {len(fold3)} patches (FULL eval — paper-grade)")

EVAL_PROMPT = "cell"
IOU_THRESH = 0.3
SCORE_THRESH = 0.3
NUM_EVAL = len(fold3)
print(f"Prompt: '{EVAL_PROMPT}'  |  IoU thresh: {IOU_THRESH}  |  N={NUM_EVAL}")

## 03 — Inference pipeline

In [ ]:
from sam3.model.data_misc import FindStage

transform = make_transform(resolution=1008)
find_stage = FindStage(
    img_ids=torch.tensor([0], device=device, dtype=torch.long),
    text_ids=torch.tensor([0], device=device, dtype=torch.long),
    input_boxes=None, input_boxes_mask=None, input_boxes_label=None,
    input_points=None, input_points_mask=None,
)

@torch.no_grad()
def run_sam3_inference(pil_img, prompt):
    """Run SAM3+LoRA, return (pred_masks_list, scores, backbone_feat)."""
    backbone_out = encode_image_frozen(model, transform, pil_img, device=device)

    feat = None
    if "vision_features" in backbone_out:
        feat = backbone_out["vision_features"]
    elif "backbone_fpn" in backbone_out:
        feat = backbone_out["backbone_fpn"][-1]
    else:
        for k, v in backbone_out.items():
            if isinstance(v, torch.Tensor) and v.dim() == 4:
                feat = v
                break
    assert feat is not None, "Cannot find backbone feature for ROI pooling"
    if feat.dim() == 4:
        feat = feat[0]

    text_out = encode_text(model, prompt, device=device)
    backbone_out.update(text_out)
    geom = model._get_dummy_prompt()
    outputs = forward_decoder_with_grad(model, backbone_out, find_stage, geom)

    pred_logits = outputs["pred_logits"].float()
    pred_masks  = outputs["pred_masks"].float()
    pres_logit  = outputs["presence_logit_dec"].float()
    cls_prob = pred_logits.sigmoid()
    pres = pres_logit.sigmoid().unsqueeze(1)
    prob = (cls_prob * pres).squeeze(-1).squeeze(0)

    masks_up = F.interpolate(
        pred_masks, size=(256, 256), mode="bilinear", align_corners=False
    ).sigmoid().squeeze(0)
    masks_bin = (masks_up > 0.5)

    keep = prob > SCORE_THRESH
    if keep.sum() == 0:
        return [], [], feat
    pred_masks_list = [masks_bin[i].cpu().numpy().astype(bool)
                       for i in range(len(masks_bin)) if keep[i]]
    scores_list = prob[keep].cpu().tolist()
    return pred_masks_list, scores_list, feat

@torch.no_grad()
def predict_types_for_image(pil_img, prompt=EVAL_PROMPT):
    """Forward SAM3+TypeHead. Returns (masks, scores, type_probs, type_logits)."""
    pred_masks_list, scores_list, backbone_feat = run_sam3_inference(pil_img, prompt)
    N = len(pred_masks_list)
    if N == 0:
        return [], [], np.zeros((0, 5)), np.zeros((0, 5))

    features = torch.zeros(N, 256, device=device)
    for i, pm in enumerate(pred_masks_list):
        mask_t = torch.from_numpy(pm).to(device)
        features[i] = roi_pool_feature(backbone_feat, mask_t)

    type_logits = type_head(features)
    type_probs = type_logits.softmax(dim=-1)
    return pred_masks_list, scores_list, type_probs.cpu().numpy(), type_logits.cpu().numpy()

print("Inference pipeline ready.")

## 04 — Full Fold 3 eval

In [ ]:
correct = 0
total = 0
per_class_correct = {c: 0 for c in CELL_TYPES}
per_class_total = {c: 0 for c in CELL_TYPES}
confusion = np.zeros((5, 5), dtype=np.int64)

counting_errors = {c: [] for c in CELL_TYPES}
pred_counts_log = {c: [] for c in CELL_TYPES}
gt_counts_log = {c: [] for c in CELL_TYPES}

n_detections_per_image = []
n_gt_per_image = []
n_matched_per_image = []
n_images_no_detection = 0
n_images_no_gt = 0

t0 = time.time()
for idx in tqdm(range(NUM_EVAL), desc="A3 eval full Fold 3"):
    sample = fold3[idx]
    pil_img = Image.fromarray(sample["image"]).convert("RGB")

    gt_masks, gt_classes = extract_gt_instances(sample, CELL_TYPES)
    gt_counts = {c: 0 for c in CELL_TYPES}
    for ci in gt_classes:
        gt_counts[CELL_TYPES[ci]] += 1
    n_gt_per_image.append(len(gt_masks))
    for c in CELL_TYPES:
        gt_counts_log[c].append(gt_counts[c])

    if len(gt_masks) == 0:
        n_images_no_gt += 1

    pred_masks_list, scores_list, type_probs, _ = predict_types_for_image(
        pil_img, prompt=EVAL_PROMPT
    )
    n_detections_per_image.append(len(pred_masks_list))

    if len(pred_masks_list) == 0:
        n_images_no_detection += 1
        for c in CELL_TYPES:
            counting_errors[c].append(gt_counts[c])
            pred_counts_log[c].append(0)
        n_matched_per_image.append(0)
        continue

    scores_arr = np.array(scores_list)
    pred_counts = per_class_counts(scores_arr, type_probs)
    for ci in range(5):
        c = CELL_TYPES[ci]
        counting_errors[c].append(abs(pred_counts[ci] - gt_counts[c]))
        pred_counts_log[c].append(float(pred_counts[ci]))

    if len(gt_masks) > 0:
        iou_matrix = compute_iou_matrix(pred_masks_list, gt_masks)
        matches = hungarian_match(iou_matrix, iou_thresh=IOU_THRESH)
        n_matched_per_image.append(len(matches))
        for pred_i, gt_j in matches:
            true_class = gt_classes[gt_j]
            pred_class = int(type_probs[pred_i].argmax())
            confusion[true_class, pred_class] += 1
            per_class_total[CELL_TYPES[true_class]] += 1
            if true_class == pred_class:
                correct += 1
                per_class_correct[CELL_TYPES[true_class]] += 1
            total += 1
    else:
        n_matched_per_image.append(0)

elapsed = time.time() - t0
print(f"\nDone {NUM_EVAL} patches in {elapsed/60:.1f} min ({elapsed/NUM_EVAL:.2f}s/patch)")

## 05 — Report

In [ ]:
A3_TRAIN_EVAL_RESULTS = {
    "Neoplastic":   89.11,
    "Inflammatory": 55.22,
    "Connective":   79.98,
    "Dead":         0.00,
    "Epithelial":   85.04,
    "Macro":        83.17,
}

macro_acc = correct / max(total, 1)

print("=" * 80)
print(f"PHASE A3 EVAL — Full Fold 3 (N={NUM_EVAL})")
print("=" * 80)
print(f"\nMacro accuracy: {macro_acc*100:.2f}%  ({correct}/{total} matched detections)")
print(f"Total matched detections: {total:,}")
print(f"Images with no detection: {n_images_no_detection}/{NUM_EVAL}")
print(f"Images with no GT:        {n_images_no_gt}/{NUM_EVAL}")
print(f"Avg detections/image:     {np.mean(n_detections_per_image):.1f}")
print(f"Avg GT/image:             {np.mean(n_gt_per_image):.1f}")
print(f"Avg matched/image:        {np.mean(n_matched_per_image):.1f}")

print("\n" + "-" * 80)
print(f"{'Class':14s} | {'A3 train (500 F1)':>20s} | {'A3 eval (full F3)':>20s} | {'delta':>8s}")
print("-" * 80)
for c in CELL_TYPES:
    if per_class_total[c] > 0:
        acc_c = per_class_correct[c] / per_class_total[c] * 100
    else:
        acc_c = 0.0
    train_acc = A3_TRAIN_EVAL_RESULTS[c]
    delta = acc_c - train_acc
    arrow = "+" if delta >= 0 else ""
    print(f"  {c:12s} | {train_acc:>17.2f}%  | {acc_c:>17.2f}%  | {arrow}{delta:>6.2f}pp")
print("-" * 80)
delta_macro = macro_acc*100 - A3_TRAIN_EVAL_RESULTS["Macro"]
arrow_m = "+" if delta_macro >= 0 else ""
print(f"  {'Macro':12s} | {A3_TRAIN_EVAL_RESULTS['Macro']:>17.2f}%  | "
      f"{macro_acc*100:>17.2f}%  | {arrow_m}{delta_macro:>6.2f}pp")

print("\n" + "-" * 80)
print("Confusion matrix (rows=true, cols=pred):")
print(f"  {'':14s} " + " ".join(f"{c[:6]:>8s}" for c in CELL_TYPES))
for i, c_true in enumerate(CELL_TYPES):
    row = " ".join(f"{confusion[i, j]:>8d}" for j in range(5))
    total_i = confusion[i].sum()
    print(f"  {c_true:14s} {row}   ({total_i} total)")

print("\n" + "-" * 80)
print("Counting MAE per class (|N_pred - N_gt| averaged over images):")
print(f"  {'Class':14s} | {'MAE':>8s} | {'Mean GT':>8s} | {'Mean Pred':>10s} | {'rel error':>10s}")
print("-" * 80)
for c in CELL_TYPES:
    if counting_errors[c]:
        mae = float(np.mean(counting_errors[c]))
        mean_gt = float(np.mean(gt_counts_log[c]))
        mean_pred = float(np.mean(pred_counts_log[c]))
        rel = mae / max(mean_gt, 1e-6) * 100
        print(f"  {c:12s} | {mae:>7.3f}  | {mean_gt:>7.2f}  | {mean_pred:>9.2f}  | {rel:>8.1f}%")

In [ ]:
print("\n" + "=" * 80)
print("PASS criteria check (paper-grade eval):")
print("=" * 80)

macro = macro_acc * 100
neo   = per_class_correct["Neoplastic"]   / max(per_class_total["Neoplastic"], 1)   * 100
inf   = per_class_correct["Inflammatory"] / max(per_class_total["Inflammatory"], 1) * 100
con   = per_class_correct["Connective"]   / max(per_class_total["Connective"], 1)   * 100
dead  = per_class_correct["Dead"]         / max(per_class_total["Dead"], 1)         * 100
epi   = per_class_correct["Epithelial"]   / max(per_class_total["Epithelial"], 1)   * 100

dead_count = per_class_total["Dead"]
neo_mae = float(np.mean(counting_errors["Neoplastic"]))

print(f"  Macro accuracy >= 60%?       {macro:.2f}%  ->  {'PASS' if macro >= 60 else 'FAIL'}")
print(f"  Neoplastic   accuracy >= 75%? {neo:.2f}%  ->  {'PASS' if neo >= 75 else 'FAIL'}")
print(f"  Epithelial   accuracy >= 70%? {epi:.2f}%  ->  {'PASS' if epi >= 70 else 'FAIL'}")
print(f"  Connective   accuracy >= 60%? {con:.2f}%  ->  {'PASS' if con >= 60 else 'FAIL'}")
print(f"  Inflammatory accuracy >= 40%? {inf:.2f}%  ->  {'PASS' if inf >= 40 else 'FAIL'}")
print(f"  Dead present (N >= 5)?        {dead_count} instances  ->  "
      f"{'PASS' if dead_count >= 5 else 'WARN (Dead rare in PanNuke Fold 3)'}")
print(f"  Counting MAE Neoplastic <= 5? {neo_mae:.2f}  ->  {'PASS' if neo_mae <= 5 else 'FAIL'}")

In [ ]:
final_results = {
    "config": {
        "num_eval": NUM_EVAL,
        "fold": 3,
        "prompt": EVAL_PROMPT,
        "iou_thresh": IOU_THRESH,
        "score_thresh": SCORE_THRESH,
        "lora_ckpt": LORA_CKPT_PATH,
        "typehead_ckpt": TYPEHEAD_CKPT_PATH,
        "elapsed_minutes": elapsed / 60,
    },
    "a3_train_eval_reference": A3_TRAIN_EVAL_RESULTS,
    "eval": {
        "macro_type_accuracy": macro_acc,
        "per_class_accuracy": {
            c: (per_class_correct[c] / max(per_class_total[c], 1))
            for c in CELL_TYPES
        },
        "per_class_total_matched": {c: per_class_total[c] for c in CELL_TYPES},
        "per_class_correct": {c: per_class_correct[c] for c in CELL_TYPES},
        "per_class_counting_mae": {
            c: float(np.mean(counting_errors[c]))
            for c in CELL_TYPES if counting_errors[c]
        },
        "per_class_mean_gt_count": {
            c: float(np.mean(gt_counts_log[c]))
            for c in CELL_TYPES if gt_counts_log[c]
        },
        "per_class_mean_pred_count": {
            c: float(np.mean(pred_counts_log[c]))
            for c in CELL_TYPES if pred_counts_log[c]
        },
        "confusion_matrix": confusion.tolist(),
        "total_matched_detections": total,
        "n_images_no_detection": n_images_no_detection,
        "n_images_no_gt": n_images_no_gt,
        "mean_detections_per_image": float(np.mean(n_detections_per_image)),
        "mean_gt_per_image": float(np.mean(n_gt_per_image)),
        "mean_matched_per_image": float(np.mean(n_matched_per_image)),
    },
}
out_path = f"{WORK}/phase_A3_final_results.json"
with open(out_path, "w") as f:
    json.dump(final_results, f, indent=2)
print(f"Saved: {out_path}")

print("\n" + "=" * 80)
print("PHASE A3 EVAL DONE — paper-grade results on full Fold 3")
print("=" * 80)
print(f"  Results JSON: {out_path}")
print(f"  Next step   : Phase C (conformal counting) uses (s_i, p_ik) ready")

## Phase A3 Eval — PASS criteria

- **Macro accuracy ≥ 60%** (relaxed from train eval 83% vì test fold)
- **Neoplastic ≥ 75%** (most common class, should be robust)
- **Epithelial ≥ 70%**
- **Connective ≥ 60%**
- **Inflammatory ≥ 40%** (challenging — small cells, similar to others)
- **Dead present** (N ≥ 5 matched) — even rare class should appear
- **Counting MAE Neoplastic ≤ 5** cells/image

**Nếu macro < 50%:** A3 train trên Fold 1+2 không transfer tốt sang Fold 3.
Cần consider:
- Cross-fold train (rotate folds)
- Feature normalization
- Test ablation: train on Fold 1 only + eval Fold 2 + 3

**Output:**
- `/kaggle/working/phase_A3_final_results.json` — paper Section 4.3 table